In [2]:
# # Phase 02 — Cross-Modal Pairing Strategy

# ## Step 02.1 — Scientifically Valid Knowledge-Level Pairing Design

# ### Objective

# Design a scientifically valid strategy for constructing cross-modal
# training relationships between Genetic, Behavioral, and Environmental
# embedding spaces.

# ### Core Methodological Principle

# This project does NOT use patient-level correspondence.

# The three modalities contain independently collected datasets with
# different individuals and substantially different sample sizes.

# Therefore:

# - No sample index matching is used.
# - No patient identity correspondence is assumed.
# - No row-to-row biological identity relationship is claimed.

# Cross-modal relationships are instead constructed at the:

# > **Knowledge level and class-semantic level**

# using independently learned 32-dimensional representations and their
# ASD / Non-ASD class information.

# ### Goal of This Notebook

# # Before generating training pairs, this notebook will:

# # 1. Define valid and invalid pairing assumptions.
# 2. Examine candidate cross-modal pairing strategies.
# 3. Quantify potential pairing-space imbalance.
# 4. Establish balance requirements.
# 5. Select a scientifically defensible knowledge-level pairing framework.

# ---

# **Project:** GAN-Enhanced Multimodal ASD Screening  
# **Phase:** Cross-Modal Learning  
# **Analysis Type:** Knowledge-Level Cross-Modal Learning  
# **Patient-Level Matching:** Not Used

In [3]:
# ============================================================
# STEP 02.1 — CROSS-MODAL PAIRING STRATEGY DESIGN
# CELL 2 — IMPORTS AND PROJECT CONFIGURATION
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd
from itertools import combinations
from datetime import datetime

print("=" * 100)
print("STEP 02.1 — CROSS-MODAL PAIRING STRATEGY DESIGN")
print("=" * 100)

# ------------------------------------------------------------
# Resolve Project Root
# ------------------------------------------------------------

PROJECT_ROOT = Path.cwd().resolve()

while PROJECT_ROOT.name != "GAN-Enhanced-Multimodal-ASD-Screening":
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        raise RuntimeError(
            "Could not locate project root "
            "'GAN-Enhanced-Multimodal-ASD-Screening'"
        )
    PROJECT_ROOT = PROJECT_ROOT.parent

# ------------------------------------------------------------
# Define Important Directories
# ------------------------------------------------------------

FUSION_DIR = PROJECT_ROOT / "05_multimodal_fusion"

PAIRING_ROOT = (
    PROJECT_ROOT
    / "06_cross_modal_learning"
    / "02_cross_modal_pairing"
)

PAIR_DIR = PAIRING_ROOT / "pairs"
REPORT_DIR = PAIRING_ROOT / "reports"

# Ensure directories exist
PAIR_DIR.mkdir(parents=True, exist_ok=True)
REPORT_DIR.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# Run Metadata
# ------------------------------------------------------------

RUN_TIMESTAMP = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

print("\nPROJECT CONFIGURATION")
print("-" * 100)

print(f"Project Root:\n{PROJECT_ROOT}")
print(f"\nFrozen Embedding Source:\n{FUSION_DIR}")
print(f"\nPair Output Directory:\n{PAIR_DIR}")
print(f"\nReport Directory:\n{REPORT_DIR}")

print(f"\nTimestamp: {RUN_TIMESTAMP}")

print("\n" + "=" * 100)
print("INITIALIZATION COMPLETED")
print("=" * 100)

STEP 02.1 — CROSS-MODAL PAIRING STRATEGY DESIGN

PROJECT CONFIGURATION
----------------------------------------------------------------------------------------------------
Project Root:
C:\Users\shasw\Documents\GitHub\GAN-Enhanced-Multimodal-ASD-Screening

Frozen Embedding Source:
C:\Users\shasw\Documents\GitHub\GAN-Enhanced-Multimodal-ASD-Screening\05_multimodal_fusion

Pair Output Directory:
C:\Users\shasw\Documents\GitHub\GAN-Enhanced-Multimodal-ASD-Screening\06_cross_modal_learning\02_cross_modal_pairing\pairs

Report Directory:
C:\Users\shasw\Documents\GitHub\GAN-Enhanced-Multimodal-ASD-Screening\06_cross_modal_learning\02_cross_modal_pairing\reports

Timestamp: 2026-09-04 17:25:05

INITIALIZATION COMPLETED


In [4]:
# ============================================================
# CELL 3 — LOAD FROZEN EMBEDDINGS AND LABELS
# ============================================================

print("=" * 100)
print("CELL 3 — LOADING FROZEN EMBEDDINGS")
print("=" * 100)

# ------------------------------------------------------------
# Define Frozen Baseline Paths
# ------------------------------------------------------------

embedding_paths = {
    "Genetic": FUSION_DIR / "genetic_32d_normalized.npy",
    "Behavioral": FUSION_DIR / "behavioral_32d_normalized.npy",
    "Environmental": FUSION_DIR / "environmental_32d_normalized.npy",
}

target_paths = {
    "Genetic": FUSION_DIR / "genetic_targets.npy",
    "Behavioral": FUSION_DIR / "behavioral_targets.npy",
    "Environmental": FUSION_DIR / "environmental_targets.npy",
}

# ------------------------------------------------------------
# Load Embeddings and Labels
# ------------------------------------------------------------

modalities = {}

for modality in ["Genetic", "Behavioral", "Environmental"]:

    embedding_path = embedding_paths[modality]
    target_path = target_paths[modality]

    # Verify files exist
    if not embedding_path.exists():
        raise FileNotFoundError(
            f"Embedding file not found: {embedding_path}"
        )

    if not target_path.exists():
        raise FileNotFoundError(
            f"Target file not found: {target_path}"
        )

    embeddings = np.load(embedding_path)
    labels = np.load(target_path)

    modalities[modality] = {
        "embeddings": embeddings,
        "labels": labels
    }

    print(f"\n{modality.upper()}")
    print("-" * 60)
    print(f"Embedding shape : {embeddings.shape}")
    print(f"Label shape     : {labels.shape}")
    print(f"Embedding dtype : {embeddings.dtype}")
    print(f"Label dtype     : {labels.dtype}")
    print(f"Unique labels   : {np.unique(labels)}")

print("\n" + "=" * 100)
print("✓ FROZEN EMBEDDINGS LOADED SUCCESSFULLY")
print("✓ NO ARTIFACTS MODIFIED")
print("=" * 100)

CELL 3 — LOADING FROZEN EMBEDDINGS

GENETIC
------------------------------------------------------------
Embedding shape : (116, 32)
Label shape     : (116,)
Embedding dtype : float32
Label dtype     : int64
Unique labels   : [0 1]

BEHAVIORAL
------------------------------------------------------------
Embedding shape : (880, 32)
Label shape     : (880,)
Embedding dtype : float32
Label dtype     : int64
Unique labels   : [0 1]

ENVIRONMENTAL
------------------------------------------------------------
Embedding shape : (47768, 32)
Label shape     : (47768,)
Embedding dtype : float32
Label dtype     : int64
Unique labels   : [0 1]

✓ FROZEN EMBEDDINGS LOADED SUCCESSFULLY
✓ NO ARTIFACTS MODIFIED


In [5]:
# ============================================================
# CELL 4 — INSPECT AVAILABLE PAIRING INFORMATION
# ============================================================

print("=" * 100)
print("CELL 4 — AVAILABLE INFORMATION FOR CROSS-MODAL PAIRING")
print("=" * 100)

pairing_information = []

for modality, data in modalities.items():

    embeddings = data["embeddings"]
    labels = data["labels"]

    unique_labels = np.unique(labels)

    pairing_information.append({
        "Modality": modality,
        "Samples": len(embeddings),
        "Embedding_Dimension": embeddings.shape[1],
        "Class_Labels_Available": list(unique_labels),
        "Embedding_Representation_Available": True,
        "Class_Label_Available": True,
        "Shared_Patient_ID_Available": False,
        "Cross_Modal_Row_Correspondence_Available": False
    })

pairing_info_df = pd.DataFrame(pairing_information)

print("\nPAIRING INFORMATION INVENTORY")
print("-" * 100)

display(pairing_info_df)

# Save report
output_path = (
    REPORT_DIR
    / "02_01_available_pairing_information.csv"
)

pairing_info_df.to_csv(output_path, index=False)

print("\n" + "=" * 100)
print("METHODOLOGICAL INTERPRETATION")
print("=" * 100)

print("""
AVAILABLE FOR PAIRING:
✓ Learned 32D representation
✓ Class information (ASD / NON-ASD)
✓ Modality identity

NOT AVAILABLE FOR PAIRING:
✗ Shared patient identity
✗ Subject correspondence
✗ Cross-modal row correspondence

Therefore:
Pairing must be based on representation-level and class-level
relationships rather than individual patient matching.
""")

print(f"Report saved:\n{output_path}")

CELL 4 — AVAILABLE INFORMATION FOR CROSS-MODAL PAIRING

PAIRING INFORMATION INVENTORY
----------------------------------------------------------------------------------------------------


,Modality,Samples,Embedding_Dimension,Class_Labels_Available,Embedding_Representation_Available,Class_Label_Available,Shared_Patient_ID_Available,Cross_Modal_Row_Correspondence_Available
0,Genetic,116,32,"[0, 1]",True,True,False,False
1,Behavioral,880,32,"[0, 1]",True,True,False,False
2,Environmental,47768,32,"[0, 1]",True,True,False,False



METHODOLOGICAL INTERPRETATION

AVAILABLE FOR PAIRING:
✓ Learned 32D representation
✓ Class information (ASD / NON-ASD)
✓ Modality identity

NOT AVAILABLE FOR PAIRING:
✗ Shared patient identity
✗ Subject correspondence
✗ Cross-modal row correspondence

Therefore:
Pairing must be based on representation-level and class-level
relationships rather than individual patient matching.

Report saved:
C:\Users\shasw\Documents\GitHub\GAN-Enhanced-Multimodal-ASD-Screening\06_cross_modal_learning\02_cross_modal_pairing\reports\02_01_available_pairing_information.csv


In [6]:
# ============================================================
# CELL 5 — DEFINE FORBIDDEN PAIRING ASSUMPTIONS
# ============================================================

print("=" * 100)
print("CELL 5 — FORBIDDEN PAIRING ASSUMPTIONS")
print("=" * 100)

forbidden_assumptions = pd.DataFrame([
    {
        "Assumption": "Same row index means same patient",
        "Allowed": False,
        "Reason": (
            "Datasets were collected independently and do not "
            "share verified patient correspondence."
        )
    },
    {
        "Assumption": "Random cross-modal samples are biologically paired",
        "Allowed": False,
        "Reason": (
            "Random samples from different modalities cannot be "
            "interpreted as measurements from the same individual."
        )
    },
    {
        "Assumption": "Sample IDs can be inferred from array position",
        "Allowed": False,
        "Reason": (
            "Array position is only a dataset-specific index and "
            "contains no cross-modal identity information."
        )
    },
    {
        "Assumption": "Different modalities represent identical populations",
        "Allowed": False,
        "Reason": (
            "Dataset sizes and class distributions differ substantially "
            "across modalities."
        )
    },
    {
        "Assumption": "Pairing establishes patient-level ground truth",
        "Allowed": False,
        "Reason": (
            "Knowledge-level pairing represents semantic relationships, "
            "not patient identity correspondence."
        )
    }
])

display(forbidden_assumptions)

# Save
output_path = (
    REPORT_DIR
    / "02_01_forbidden_pairing_assumptions.csv"
)

forbidden_assumptions.to_csv(output_path, index=False)

print("\n" + "=" * 100)
print("✓ PAIRING SAFETY RULES DOCUMENTED")
print("=" * 100)

print(f"\nReport saved:\n{output_path}")

CELL 5 — FORBIDDEN PAIRING ASSUMPTIONS


,Assumption,Allowed,Reason
0,Same row index means same patient,False,Datasets were collected independently and do n...
1,Random cross-modal samples are biologically pa...,False,Random samples from different modalities canno...
2,Sample IDs can be inferred from array position,False,Array position is only a dataset-specific inde...
3,Different modalities represent identical popul...,False,Dataset sizes and class distributions differ s...
4,Pairing establishes patient-level ground truth,False,Knowledge-level pairing represents semantic re...



✓ PAIRING SAFETY RULES DOCUMENTED

Report saved:
C:\Users\shasw\Documents\GitHub\GAN-Enhanced-Multimodal-ASD-Screening\06_cross_modal_learning\02_cross_modal_pairing\reports\02_01_forbidden_pairing_assumptions.csv


In [7]:
# ============================================================
# CELL 6 — CANDIDATE CROSS-MODAL PAIRING STRATEGIES
# ============================================================

print("=" * 100)
print("CELL 6 — CANDIDATE PAIRING STRATEGIES")
print("=" * 100)

strategy_comparison = pd.DataFrame([
    {
        "Strategy": "Unconditional Random Pairing",
        "Description": (
            "Randomly pair samples across modalities without "
            "considering class labels."
        ),
        "Knowledge_Level_Valid": False,
        "Preserves_Class_Semantics": False,
        "Patient_Assumption": False,
        "Primary_Concern": (
            "Pairs may connect semantically unrelated ASD and "
            "NON-ASD representations."
        )
    },
    {
        "Strategy": "Class-Conditional Random Pairing",
        "Description": (
            "Pair samples across modalities only when they share "
            "the same diagnostic class."
        ),
        "Knowledge_Level_Valid": True,
        "Preserves_Class_Semantics": True,
        "Patient_Assumption": False,
        "Primary_Concern": (
            "Requires balancing to prevent dominant classes "
            "or modalities from controlling training."
        )
    },
    {
        "Strategy": "Class Centroid Pairing",
        "Description": (
            "Pair aggregate class centroids across modality "
            "representation spaces."
        ),
        "Knowledge_Level_Valid": True,
        "Preserves_Class_Semantics": True,
        "Patient_Assumption": False,
        "Primary_Concern": (
            "Removes individual representation diversity."
        )
    },
    {
        "Strategy": "Prototype / Distribution-Aware Pairing",
        "Description": (
            "Select representative samples or prototypes and pair "
            "them based on class-level distribution structure."
        ),
        "Knowledge_Level_Valid": True,
        "Preserves_Class_Semantics": True,
        "Patient_Assumption": False,
        "Primary_Concern": (
            "More complex and requires additional selection criteria."
        )
    }
])

display(strategy_comparison)

output_path = (
    REPORT_DIR
    / "02_01_candidate_pairing_strategies.csv"
)

strategy_comparison.to_csv(output_path, index=False)

print("\n" + "=" * 100)
print("✓ CANDIDATE PAIRING STRATEGIES DOCUMENTED")
print("=" * 100)

print(f"\nReport saved:\n{output_path}")

CELL 6 — CANDIDATE PAIRING STRATEGIES


,Strategy,Description,Knowledge_Level_Valid,Preserves_Class_Semantics,Patient_Assumption,Primary_Concern
0,Unconditional Random Pairing,Randomly pair samples across modalities withou...,False,False,False,Pairs may connect semantically unrelated ASD a...
1,Class-Conditional Random Pairing,Pair samples across modalities only when they ...,True,True,False,Requires balancing to prevent dominant classes...
2,Class Centroid Pairing,Pair aggregate class centroids across modality...,True,True,False,Removes individual representation diversity.
3,Prototype / Distribution-Aware Pairing,Select representative samples or prototypes an...,True,True,False,More complex and requires additional selection...



✓ CANDIDATE PAIRING STRATEGIES DOCUMENTED

Report saved:
C:\Users\shasw\Documents\GitHub\GAN-Enhanced-Multimodal-ASD-Screening\06_cross_modal_learning\02_cross_modal_pairing\reports\02_01_candidate_pairing_strategies.csv


In [8]:
# ============================================================
# CELL 7 — QUANTIFY CROSS-MODAL PAIR SPACE
# ============================================================

print("=" * 100)
print("CELL 7 — CROSS-MODAL PAIR SPACE ANALYSIS")
print("=" * 100)

pair_space_results = []

modality_pairs = list(
    combinations(
        ["Genetic", "Behavioral", "Environmental"],
        2
    )
)

classes = [0, 1]

for modality_a, modality_b in modality_pairs:

    labels_a = modalities[modality_a]["labels"]
    labels_b = modalities[modality_b]["labels"]

    for class_label in classes:

        count_a = np.sum(labels_a == class_label)
        count_b = np.sum(labels_b == class_label)

        possible_pairs = int(count_a * count_b)

        pair_space_results.append({
            "Modality_A": modality_a,
            "Modality_B": modality_b,
            "Class": class_label,
            "Samples_A": int(count_a),
            "Samples_B": int(count_b),
            "Possible_Same_Class_Pairs": possible_pairs
        })

pair_space_df = pd.DataFrame(pair_space_results)

print("\nPOTENTIAL SAME-CLASS PAIR SPACE")
print("-" * 100)

display(pair_space_df)

print("\nTOTAL POSSIBLE SAME-CLASS PAIRS")
print("-" * 100)

total_pairs = pair_space_df[
    "Possible_Same_Class_Pairs"
].sum()

print(f"{total_pairs:,}")

# Save report
output_path = (
    REPORT_DIR
    / "02_01_pair_space_analysis.csv"
)

pair_space_df.to_csv(output_path, index=False)

print("\n" + "=" * 100)
print("INTERPRETATION")
print("=" * 100)

print("""
This calculation shows how many potential same-class relationships
exist between each modality pair.

We do NOT automatically generate all of these combinations.

The purpose is to identify:

1. Computational pair explosion.
2. Dataset-size dominance.
3. Class imbalance influence.

The results will guide a controlled and balanced pair-generation
strategy in the next stage.
""")

print(f"\nReport saved:\n{output_path}")

CELL 7 — CROSS-MODAL PAIR SPACE ANALYSIS

POTENTIAL SAME-CLASS PAIR SPACE
----------------------------------------------------------------------------------------------------


,Modality_A,Modality_B,Class,Samples_A,Samples_B,Possible_Same_Class_Pairs
0,Genetic,Behavioral,0,29,439,12731
1,Genetic,Behavioral,1,87,441,38367
2,Genetic,Environmental,0,29,45650,1323850
3,Genetic,Environmental,1,87,2118,184266
4,Behavioral,Environmental,0,439,45650,20040350
5,Behavioral,Environmental,1,441,2118,934038



TOTAL POSSIBLE SAME-CLASS PAIRS
----------------------------------------------------------------------------------------------------
22,533,602

INTERPRETATION

This calculation shows how many potential same-class relationships
exist between each modality pair.

We do NOT automatically generate all of these combinations.

The purpose is to identify:

1. Computational pair explosion.
2. Dataset-size dominance.
3. Class imbalance influence.

The results will guide a controlled and balanced pair-generation
strategy in the next stage.


Report saved:
C:\Users\shasw\Documents\GitHub\GAN-Enhanced-Multimodal-ASD-Screening\06_cross_modal_learning\02_cross_modal_pairing\reports\02_01_pair_space_analysis.csv


In [9]:
# ============================================================
# CELL 8 — PROPOSED KNOWLEDGE-LEVEL PAIRING FRAMEWORK
# ============================================================

print("=" * 100)
print("CELL 8 — PROPOSED CROSS-MODAL PAIRING FRAMEWORK")
print("=" * 100)

pairing_framework = pd.DataFrame([
    {
        "Component": "Learning Level",
        "Decision": "Knowledge-Level / Representation-Level",
        "Justification": (
            "No verified patient correspondence exists across modalities."
        )
    },
    {
        "Component": "Positive Pair Definition",
        "Decision": "Same-Class Cross-Modal Relationship",
        "Justification": (
            "Pairs share diagnostic class semantics without claiming "
            "individual identity correspondence."
        )
    },
    {
        "Component": "Negative Pair Definition",
        "Decision": "Different-Class Cross-Modal Relationship",
        "Justification": (
            "Provides controlled semantic contrast between ASD and NON-ASD "
            "representation knowledge."
        )
    },
    {
        "Component": "Modality Balance",
        "Decision": "Equal Controlled Contribution",
        "Justification": (
            "Prevents large datasets from dominating cross-modal learning."
        )
    },
    {
        "Component": "Class Balance",
        "Decision": "Controlled Class-Conditional Sampling",
        "Justification": (
            "Prevents majority classes from dominating pair generation."
        )
    },
    {
        "Component": "Pair Generation",
        "Decision": "Sampled Rather Than Exhaustive Cartesian Product",
        "Justification": (
            "Avoids pair-space explosion and repeated-pair bias."
        )
    },
    {
        "Component": "Patient-Level Matching",
        "Decision": "Explicitly Prohibited",
        "Justification": (
            "Datasets contain independent populations without shared "
            "patient identity correspondence."
        )
    }
])

display(pairing_framework)

output_path = (
    REPORT_DIR
    / "02_01_proposed_pairing_framework.csv"
)

pairing_framework.to_csv(output_path, index=False)

print("\n" + "=" * 100)
print("PROPOSED METHODOLOGICAL DECISION")
print("=" * 100)

print("""
PROPOSED STRATEGY:

BALANCED CLASS-CONDITIONAL KNOWLEDGE-LEVEL PAIRING

Positive relationships:
    ASD ↔ ASD
    NON-ASD ↔ NON-ASD

Negative relationships:
    ASD ↔ NON-ASD
    NON-ASD ↔ ASD

Critical rule:
These pairs represent relationships between learned knowledge
representations.

They DO NOT represent paired measurements from the same patient.
""")

print(f"\nFramework saved:\n{output_path}")

CELL 8 — PROPOSED CROSS-MODAL PAIRING FRAMEWORK


,Component,Decision,Justification
0,Learning Level,Knowledge-Level / Representation-Level,No verified patient correspondence exists acro...
1,Positive Pair Definition,Same-Class Cross-Modal Relationship,Pairs share diagnostic class semantics without...
2,Negative Pair Definition,Different-Class Cross-Modal Relationship,Provides controlled semantic contrast between ...
3,Modality Balance,Equal Controlled Contribution,Prevents large datasets from dominating cross-...
4,Class Balance,Controlled Class-Conditional Sampling,Prevents majority classes from dominating pair...
5,Pair Generation,Sampled Rather Than Exhaustive Cartesian Product,Avoids pair-space explosion and repeated-pair ...
6,Patient-Level Matching,Explicitly Prohibited,Datasets contain independent populations witho...



PROPOSED METHODOLOGICAL DECISION

PROPOSED STRATEGY:

BALANCED CLASS-CONDITIONAL KNOWLEDGE-LEVEL PAIRING

Positive relationships:
    ASD ↔ ASD
    NON-ASD ↔ NON-ASD

Negative relationships:
    ASD ↔ NON-ASD
    NON-ASD ↔ ASD

Critical rule:
These pairs represent relationships between learned knowledge
representations.

They DO NOT represent paired measurements from the same patient.


Framework saved:
C:\Users\shasw\Documents\GitHub\GAN-Enhanced-Multimodal-ASD-Screening\06_cross_modal_learning\02_cross_modal_pairing\reports\02_01_proposed_pairing_framework.csv


In [10]:
# ============================================================
# CELL 9 — PAIR GENERATION BALANCE REQUIREMENTS
# ============================================================

print("=" * 100)
print("CELL 9 — PAIR GENERATION BALANCE REQUIREMENTS")
print("=" * 100)

balance_requirements = pd.DataFrame([
    {
        "Requirement": "Modality Pair Balance",
        "Rule": (
            "Genetic-Behavioral, Genetic-Environmental, and "
            "Behavioral-Environmental pairs should have controlled "
            "and comparable representation."
        )
    },
    {
        "Requirement": "Class Balance",
        "Rule": (
            "ASD and NON-ASD semantic relationships should be "
            "controlled to prevent majority-class dominance."
        )
    },
    {
        "Requirement": "Positive / Negative Balance",
        "Rule": (
            "Positive same-class and negative different-class "
            "relationships should be explicitly controlled."
        )
    },
    {
        "Requirement": "Sampling Reproducibility",
        "Rule": (
            "All randomized pair generation must use fixed "
            "random seeds."
        )
    },
    {
        "Requirement": "No Cartesian Explosion",
        "Rule": (
            "Do not generate every possible cross-product pair."
        )
    },
    {
        "Requirement": "No Patient Matching",
        "Rule": (
            "Pair indices must never be interpreted as patient "
            "identity correspondence."
        )
    },
    {
        "Requirement": "Pair Metadata",
        "Rule": (
            "Every generated pair must retain modality, class, "
            "relationship type, and source index metadata."
        )
    }
])

display(balance_requirements)

output_path = (
    REPORT_DIR
    / "02_01_pair_generation_requirements.csv"
)

balance_requirements.to_csv(output_path, index=False)

print("\n" + "=" * 100)
print("✓ PAIR GENERATION REQUIREMENTS ESTABLISHED")
print("=" * 100)

print(f"\nRequirements saved:\n{output_path}")

CELL 9 — PAIR GENERATION BALANCE REQUIREMENTS


,Requirement,Rule
0,Modality Pair Balance,"Genetic-Behavioral, Genetic-Environmental, and..."
1,Class Balance,ASD and NON-ASD semantic relationships should ...
2,Positive / Negative Balance,Positive same-class and negative different-cla...
3,Sampling Reproducibility,All randomized pair generation must use fixed ...
4,No Cartesian Explosion,Do not generate every possible cross-product p...
5,No Patient Matching,Pair indices must never be interpreted as pati...
6,Pair Metadata,"Every generated pair must retain modality, cla..."



✓ PAIR GENERATION REQUIREMENTS ESTABLISHED

Requirements saved:
C:\Users\shasw\Documents\GitHub\GAN-Enhanced-Multimodal-ASD-Screening\06_cross_modal_learning\02_cross_modal_pairing\reports\02_01_pair_generation_requirements.csv


In [11]:
# ============================================================
# CELL 10 — FINAL PAIRING STRATEGY DESIGN CONCLUSION
# ============================================================

print("=" * 100)
print("STEP 02.1 — FINAL PAIRING STRATEGY DESIGN CONCLUSION")
print("=" * 100)

final_decision = {
    "Analysis_Level": "Knowledge-Level Cross-Modal Learning",
    "Patient_Level_Matching": "Not Used",
    "Shared_Patient_Correspondence": "Not Available / Not Assumed",
    "Embedding_Space": "32-Dimensional Learned Representations",
    "Positive_Pairs": "Same-Class Cross-Modal Relationships",
    "Negative_Pairs": "Different-Class Cross-Modal Relationships",
    "Pairing_Strategy": "Balanced Class-Conditional Sampling",
    "Exhaustive_Cartesian_Product": "Not Used",
    "Modality_Balance": "Required",
    "Class_Balance": "Required",
    "Sampling_Reproducibility": "Fixed Random Seed Required",
    "Pair_Metadata": "Required",
    "Scientific_Interpretation": (
        "Pairs represent semantic relationships between independently "
        "learned modality knowledge spaces, not paired patients."
    )
}

final_decision_df = pd.DataFrame(
    list(final_decision.items()),
    columns=["Methodological_Component", "Decision"]
)

display(final_decision_df)

output_path = (
    REPORT_DIR
    / "02_01_final_pairing_strategy_decision.csv"
)

final_decision_df.to_csv(output_path, index=False)

print("\n" + "=" * 100)
print("FINAL SCIENTIFIC DECISION")
print("=" * 100)

print("""
SELECTED METHODOLOGY:

BALANCED CLASS-CONDITIONAL KNOWLEDGE-LEVEL CROSS-MODAL PAIRING

The pairing framework will:

✓ Use independently learned 32D representations
✓ Preserve modality-specific sample identity internally
✓ Use diagnostic class as semantic supervision
✓ Construct same-class knowledge relationships
✓ Construct controlled different-class relationships
✓ Balance modality contributions
✓ Balance class contributions
✓ Avoid exhaustive Cartesian pair generation
✓ Preserve reproducibility using fixed random seeds

The pairing framework will NOT:

✗ Match patients
✗ Assume shared subject identities
✗ Match samples using row indices
✗ Claim biological correspondence between paired samples

SCIENTIFIC INTERPRETATION:

A generated pair represents a training relationship between two
independently learned knowledge representations from different
modalities.

It does not represent two measurements obtained from the same
individual.
""")

print("\n" + "=" * 100)
print("✓ STEP 02.1 — PAIRING STRATEGY DESIGN COMPLETED")
print("✓ KNOWLEDGE-LEVEL METHODOLOGY PRESERVED")
print("✓ READY FOR STEP 02.2 — PAIRING STRATEGY VALIDATION")
print("=" * 100)

print(f"\nFinal decision saved:\n{output_path}")

STEP 02.1 — FINAL PAIRING STRATEGY DESIGN CONCLUSION


,Methodological_Component,Decision
0,Analysis_Level,Knowledge-Level Cross-Modal Learning
1,Patient_Level_Matching,Not Used
2,Shared_Patient_Correspondence,Not Available / Not Assumed
3,Embedding_Space,32-Dimensional Learned Representations
4,Positive_Pairs,Same-Class Cross-Modal Relationships
5,Negative_Pairs,Different-Class Cross-Modal Relationships
6,Pairing_Strategy,Balanced Class-Conditional Sampling
7,Exhaustive_Cartesian_Product,Not Used
8,Modality_Balance,Required
9,Class_Balance,Required



FINAL SCIENTIFIC DECISION

SELECTED METHODOLOGY:

BALANCED CLASS-CONDITIONAL KNOWLEDGE-LEVEL CROSS-MODAL PAIRING

The pairing framework will:

✓ Use independently learned 32D representations
✓ Preserve modality-specific sample identity internally
✓ Use diagnostic class as semantic supervision
✓ Construct same-class knowledge relationships
✓ Construct controlled different-class relationships
✓ Balance modality contributions
✓ Balance class contributions
✓ Avoid exhaustive Cartesian pair generation
✓ Preserve reproducibility using fixed random seeds

The pairing framework will NOT:

✗ Match patients
✗ Assume shared subject identities
✗ Match samples using row indices
✗ Claim biological correspondence between paired samples

SCIENTIFIC INTERPRETATION:

A generated pair represents a training relationship between two
independently learned knowledge representations from different
modalities.

It does not represent two measurements obtained from the same
individual.


✓ STEP 02.1 — PAIRING S